In [1]:
import pandas as pd
import numpy as np

np.random.seed(42)

# Hourly timestamps for 30 days
date_range = pd.date_range(start="2025-02-01", end="2025-03-02 23:00", freq="H")

intersections = ["INT_101", "INT_102", "INT_103", "INT_104"]

rows = []
for dt in date_range:
    hour = dt.hour
    for intersection in intersections:
        base_traffic = np.random.poisson(80)

        # Peak hour effects
        if 7 <= hour <= 9:
            base_traffic += 70
        elif 17 <= hour <= 19:
            base_traffic += 90
        elif 12 <= hour <= 13:
            base_traffic += 30

        # Intersection-specific patterns
        if intersection == "INT_101":
            base_traffic += 20
        elif intersection == "INT_104":
            base_traffic -= 10

        avg_speed = np.random.normal(42, 6)
        if base_traffic > 150:
            avg_speed -= np.random.uniform(5, 12)

        green_time = np.random.randint(30, 91)
        red_time = np.random.randint(30, 121)

        # Wait time affected by traffic and signal timing
        avg_wait_time = (
            20
            + (base_traffic / 12)
            + (red_time / 8)
            - (green_time / 10)
            + np.random.normal(0, 4)
        )

        incident_flag = np.random.choice([0, 1], p=[0.94, 0.06])

        if incident_flag == 1:
            avg_wait_time += np.random.uniform(10, 25)
            avg_speed -= np.random.uniform(4, 10)

        rows.append({
            "datetime": dt,
            "intersection_id": intersection,
            "vehicles_count": max(10, round(base_traffic)),
            "avg_speed": round(max(5, avg_speed), 2),
            "green_time": green_time,
            "red_time": red_time,
            "avg_wait_time": round(max(5, avg_wait_time), 2),
            "incident_flag": incident_flag
        })

df = pd.DataFrame(rows)

traffic_flow = df[["datetime", "intersection_id", "vehicles_count", "avg_speed"]]
signal_timing = df[["datetime", "intersection_id", "green_time", "red_time"]]
waiting_time = df[["datetime", "intersection_id", "avg_wait_time"]]
incidents = df[["datetime", "intersection_id", "incident_flag"]]

traffic_flow.to_csv("traffic_flow.csv", index=False)
signal_timing.to_csv("signal_timing.csv", index=False)
waiting_time.to_csv("waiting_time.csv", index=False)
incidents.to_csv("incidents.csv", index=False)

print("Files created successfully.")

Files created successfully.


In [2]:
import pandas as pd
import sqlite3

conn = sqlite3.connect("traffic_signals.db")

traffic_flow = pd.read_csv("traffic_flow.csv")
signal_timing = pd.read_csv("signal_timing.csv")
waiting_time = pd.read_csv("waiting_time.csv")
incidents = pd.read_csv("incidents.csv")

traffic_flow.to_sql("traffic_flow", conn, if_exists="replace", index=False)
signal_timing.to_sql("signal_timing", conn, if_exists="replace", index=False)
waiting_time.to_sql("waiting_time", conn, if_exists="replace", index=False)
incidents.to_sql("incidents", conn, if_exists="replace", index=False)

print("Database created.")

Database created.
